##### Clean and combine the three datasets
##### This dataset contains the Customers and their Sales and Service details from a fictitious Dealer
Note: <br> 
* The data is available in 3 different sheets of an excel file
* Customer ID needs to be split out from Customer Name
* All 3 datasets have a few Customer IDs that are not found in the other datasets
* Some Customers are also mapped into different Segmentation groups, this needs to be cleaned so that each Customer is part of a single Segmentation
* Some Sales numbers are negative. **Assumption being made:** _The Customer had a credit which was applied to the Sales transaction_
* Each dataset has multiple rows for the same Customer ID
* Information from the 3 datasets need to be combined before any insights or recommendations can be derived

In [94]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

from functools import reduce 

In [95]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  # Ignore the unnecessary noise of future warnings in the output

In [96]:
os.chdir("../Data")
# os.getcwd()
# Read each sheet of the input excel file into a dataframe

In [97]:
custdf = pd.read_excel("Vehicle Analysis.xlsx", sheet_name='Customer Opportunity')
vehicledf = pd.read_excel("Vehicle Analysis.xlsx", sheet_name='Vehicle units')
salesdf = pd.read_excel("Vehicle Analysis.xlsx", sheet_name='Sales channel')
# Vehicle Analysis

In [98]:
# Validate the Stats of the numerical columns in the dataset
custdf.describe(include="all")

,Salesman,Customer ID,Segmentation,City,Undercarriage Opportunity,Undercarriage Sales,Engine Opportunity,Engine Sales,GET Opportunity,GET Sales,...,Filters & Fluids Opportunity,Filters & Fluids Sales,Maintenance Parts and Supplies Opportunity,Maintenance Parts and Supplies Sales,"Structural, Appearance, and Other Parts Opportunity","Structural, Appearance, and Other Parts Sales",Labor Opportunity,Labor Sales,Parts Sales,Parts Opportunity
count,10165,10165,10165,10165,10165.00000,10165.000000,10165.000000,10165.000000,10165.000000,10165.000000,...,10165.000000,10165.000000,10165.000000,10165.000000,10165.000000,10165.000000,10165.000000,10165.000000,1.016500e+04,1.016500e+04
unique,78,7859,4,9,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,SALESMAN RECORD NOT FOUND - ZZZ,250250,Do It Myself,03 ADANA,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,3305,9,6479,2246,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,731.80423,327.109297,1509.608854,1286.408460,583.739990,330.609838,...,1138.013478,466.265322,604.872996,202.477127,265.515101,404.430103,2998.617216,1105.521200,4.221376e+03,7.157269e+03
std,NaN,NaN,NaN,NaN,6697.50300,6443.230580,9093.277877,12549.638641,3798.964031,5309.452956,...,6578.195420,4485.793631,3669.597705,2080.836691,1830.222281,4599.238828,15717.242113,8626.790871,4.322648e+04,4.366492e+04
min,NaN,NaN,NaN,NaN,0.00000,0.000000,0.000000,-908.000000,0.000000,-2042.000000,...,0.000000,-26.000000,0.000000,-1382.000000,0.000000,-301.000000,0.000000,0.000000,-1.835000e+03,0.000000e+00
25%,NaN,NaN,NaN,NaN,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
50%,NaN,NaN,NaN,NaN,0.00000,0.000000,235.000000,0.000000,20.000000,0.000000,...,210.000000,0.000000,68.000000,0.000000,19.000000,0.000000,573.000000,0.000000,0.000000e+00,1.132000e+03
75%,NaN,NaN,NaN,NaN,1.00000,0.000000,960.000000,24.000000,340.000000,0.000000,...,735.000000,19.000000,348.000000,0.000000,124.000000,0.000000,2138.000000,0.000000,2.780000e+02,4.532000e+03


In [99]:
custdf.head()

,Salesman,Customer ID,Segmentation,City,Undercarriage Opportunity,Undercarriage Sales,Engine Opportunity,Engine Sales,GET Opportunity,GET Sales,...,Filters & Fluids Opportunity,Filters & Fluids Sales,Maintenance Parts and Supplies Opportunity,Maintenance Parts and Supplies Sales,"Structural, Appearance, and Other Parts Opportunity","Structural, Appearance, and Other Parts Sales",Labor Opportunity,Labor Sales,Parts Sales,Parts Opportunity
0,ABDULLAH AVNIABDIOGLU - 691,C531599,Do It Myself,01 ANKARA,0,0,485,0,145,0,...,357,0,162,0,68,0,1335,0,0,2399
1,AHMET SUHAKOCAK - 223,4013245,Do It Myself,43 DIYARBAKIR,0,0,14,0,0,0,...,180,0,57,0,6,0,78,0,0,288
2,AHMET SUHAKOCAK - 223,C528031,Do It Myself,03 ADANA,698,0,627,0,518,0,...,600,0,218,0,55,0,1125,0,0,3576
3,AHMET SUHAKOCAK - 223,310480,No Product Segment Assigned,03 ADANA,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,AHMET SUHAKOCAK - 223,A00323,Do It Myself,03 ADANA,0,0,510,0,200,0,...,470,0,210,0,50,0,1660,0,0,2690


In [100]:
vehicledf.describe(include="all")

,Customer Name,Equipment units
count,10165,10165.000000
unique,9533,NaN
top,EZZUZUL IL UZEL IDEZE LUDUZL - 250250,NaN
freq,3,NaN
mean,NaN,2.259616
std,NaN,6.819999
min,NaN,0.000000
25%,NaN,0.000000
50%,NaN,1.000000
75%,NaN,2.000000


In [101]:
vehicledf.head(20)

,Customer Name,Equipment units
0,EFUZ GELIZLIH IHLEZI GIDE - C531599,1
1,EZ-DEN YUL INHEEG GEEH.GUZ. - 4013245,2
2,HEGIPUGLU INH GEEH HEN VE GI - C528031,1
3,LICIUGULLEZI INH.GIC.LGD.HGI - 310480,1
4,G.C. HEHENPEYLI - E00323,1
5,G.C.ENGEHYE PLD.PHH. - 310340,3
6,G.C.GUHHU PELEDIYEHI - E00308,1
7,HEHINUGULLEZI GICEZEG - E03987,2
8,PUZ-EN INHEEG LEHINE - C511657,8
9,PUZUHEN LEHINE VE GUC HIHGEL - PUZ073,2


In [102]:
salesdf.describe(include="all")

,customer name,Work Order sales,Over the Counter Sales
count,6660,6.660000e+03,6.660000e+03
unique,6660,NaN,NaN
top,Ell CuHGULeZ GZUupH,NaN,NaN
freq,1,NaN,NaN
mean,NaN,5.588740e+03,6.312835e+03
std,NaN,2.287109e+05,2.597358e+05
min,NaN,0.000000e+00,-2.056000e+03
25%,NaN,0.000000e+00,0.000000e+00
50%,NaN,0.000000e+00,0.000000e+00
75%,NaN,4.190000e+02,2.290000e+02


In [103]:
salesdf.head(20)

,customer name,Work Order sales,Over the Counter Sales
0,EFUZ GELIZLIH IHLEZI GIDE - C531599 - G,0,0
1,EHDEGLEZ NIHEL LED.HEN.VE GI - C527998 - G,0,0
2,DIZELCILEZ UGULUGIVGIC.VE HE - 310650 - G,0,0
3,FIZEG INHEEG DEHUZEHYUN NEH. - C528040 - G,0,0
4,HEGIPUGLU INH GEEH HEN VE GI - C528031 - G,0,0
5,EYGEHIN-HEZUL INHEEG - E03985 - G,0,0
6,PUZ-EN INHEEG LEHINE - C511657 - G,0,0
7,DEZLE HIVI ICEC.GEH.GIDE - C506340 - G,0,0
8,EHHE LED. NEH. HEF. HEN. VE - E06734 - G,3813,0
9,GUNDEL PEG.INH.NEH.HEF.GEEH. - C507732 - G,0,0


In [104]:
#Extract Customer ID and Customer Name from Vehicle and Sales dataframes

# vehicledf['Customer ID'] = vehicledf['Customer Name'].str.split(delimiter).str[-2].str.strip()

# Function to identify the Customer ID from Customer name

def split_cust_name(x):
    delimiter = "-"
    if x.find(delimiter) != -1:
        cust_id_1 = x.split(delimiter)[-1].strip()
        cust_id_2 = x.split(delimiter)[-2].strip()
        
        if len(cust_id_1) == 1 and len(cust_id_2) == 1:
            cust_id = x.split(delimiter)[-3].strip()
            return cust_id
        elif len(cust_id_1) == 1 and len(cust_id_2) != 1:
            return cust_id_2
        else:
            return cust_id_1
    else:
        return "No ID"

In [105]:
vehicledf['Customer ID'] = vehicledf['Customer Name'].apply(split_cust_name)
#vehicledf['Customer ID'].str.len().unique()
vehicledf

,Customer Name,Equipment units,Customer ID
0,EFUZ GELIZLIH IHLEZI GIDE - C531599,1,C531599
1,EZ-DEN YUL INHEEG GEEH.GUZ. - 4013245,2,4013245
2,HEGIPUGLU INH GEEH HEN VE GI - C528031,1,C528031
3,LICIUGULLEZI INH.GIC.LGD.HGI - 310480,1,310480
4,G.C. HEHENPEYLI - E00323,1,E00323
...,...,...,...
10160,LEDENCILIH LGD.HGI - 200100,1,200100
10161,UZDENLEZ LEDEN HEN.GIC.LGD.H - 200120,3,200120
10162,G.C. DENIZLI - 200070,10,200070
10163,G.C. HEZH - 360020,3,360020


In [106]:
vehicledf.info()
vehicledf['Customer ID'].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10165 entries, 0 to 10164
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Customer Name    10165 non-null  object
 1   Equipment units  10165 non-null  int64 
 2   Customer ID      10165 non-null  object
dtypes: int64(1), object(2)
memory usage: 238.4+ KB


count      10165
unique      7859
top       250250
freq           9
Name: Customer ID, dtype: object

In [107]:
salesdf['Customer ID'] = salesdf['customer name'].apply(split_cust_name)
# salesdf[salesdf['Customer ID'] == "No ID"]  # Only one "No ID" record
# salesdf['Customer ID'].str.len().unique()
salesdf

,customer name,Work Order sales,Over the Counter Sales,Customer ID
0,EFUZ GELIZLIH IHLEZI GIDE - C531599 - G,0,0,C531599
1,EHDEGLEZ NIHEL LED.HEN.VE GI - C527998 - G,0,0,C527998
2,DIZELCILEZ UGULUGIVGIC.VE HE - 310650 - G,0,0,310650
3,FIZEG INHEEG DEHUZEHYUN NEH. - C528040 - G,0,0,C528040
4,HEGIPUGLU INH GEEH HEN VE GI - C528031 - G,0,0,C528031
...,...,...,...,...
6655,LEHLEG HELIH EGEHUY - C526607 - G,0,0,C526607
6656,DENIZLI IL UZEL IDEZEHI - C522705 - G,0,0,C522705
6657,G.C. DENIZLI - 200070 - G,13998,720,200070
6658,G.C. HEZH - 360020 - G,0,0,360020


In [108]:
salesdf.info()
salesdf['Customer ID'].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6660 entries, 0 to 6659
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   customer name           6660 non-null   object
 1   Work Order sales        6660 non-null   int64 
 2   Over the Counter Sales  6660 non-null   int64 
 3   Customer ID             6660 non-null   object
dtypes: int64(2), object(2)
memory usage: 208.3+ KB


count        6660
unique       5242
top       C522972
freq            3
Name: Customer ID, dtype: object

In [109]:
# Group by Customer ID, number of records
# df.groupby('grouping_column')['column_to_count'].nunique()

#custdf.groupby('Customer ID').size()  # groupby count
#custdf.groupby('Customer ID').filter(lambda x: len(x) > 1)

custdf[custdf.groupby('Customer ID')['Segmentation'].transform('nunique') > 1][['Customer ID', 'Segmentation', 'Salesman']].sort_values(by = 'Customer ID')
# Same customer is assigned to multiple Segments

# custdf[custdf.groupby('Customer ID')['City'].transform('nunique') > 1][['Customer ID', 'City']].sort_values(by = 'Customer ID')
# Customer ID/City is a unique combination

,Customer ID,Segmentation,Salesman
6160,010030,No Product Segment Assigned,SALESMAN RECORD NOT FOUND - ZZZ
9098,010030,Do It Myself,SAMIOZDEMIR - 675
330,010130,Do It For Me,ANILYUVACI - 485
6794,010130,Do It Myself,SALESMAN RECORD NOT FOUND - ZZZ
6793,010130,Do It Myself,SALESMAN RECORD NOT FOUND - ZZZ
...,...,...,...
6459,C531871,No Product Segment Assigned,SALESMAN RECORD NOT FOUND - ZZZ
3494,C531888,Do It Myself,HAKANKILIC - 972
7891,C531888,No Product Segment Assigned,SALESMAN RECORD NOT FOUND - ZZZ
6992,C531906,Do It Myself,SALESMAN RECORD NOT FOUND - ZZZ


In [110]:
# custdf[custdf['Customer ID'] == 'C531906']
# custdf['Segmentation'].unique()
# custdf.groupby('Segmentation').size()

# Assign a rank to Segmentation, to cleanup when the same customer belongs to multiple Segmentation

condlist = [
    custdf['Segmentation'] == 'Do It For Me',
    custdf['Segmentation'] == 'Do It Myself',
    custdf['Segmentation'] == 'No Product Segment Assigned',
    custdf['Segmentation'] == 'Work With Me'
]

choicelist = [
    2, 1, 4,3
] 
    
custdf['Segmentation Rank'] = np.select(condlist, choicelist, default=99)
#custdf[['Segmentation', 'Segmentation Rank']].drop_duplicates()

# Customer Dimension
customerdf = custdf[['Customer ID', 'City', 'Segmentation', 'Segmentation Rank']].copy()
customerdf['Rk'] = pd.to_numeric(customerdf.groupby(['Customer ID', 'City'])['Segmentation Rank'].rank(method='first'))

# Cleaning up Customer IDs that had multiple Segmentation associated
# One Customer ID will belong to one Segmentation
customerdf = customerdf[customerdf['Rk'] == 1].reset_index(drop=True)
customerdf.drop(['Rk', 'Segmentation Rank'], axis=1, inplace=True)
customerdf


,Customer ID,City,Segmentation
0,C531599,01 ANKARA,Do It Myself
1,4013245,43 DIYARBAKIR,Do It Myself
2,C528031,03 ADANA,Do It Myself
3,310480,03 ADANA,No Product Segment Assigned
4,A00323,03 ADANA,Do It Myself
...,...,...,...
7854,200100,02 IZMIR,Do It Myself
7855,200120,02 IZMIR,Do It Myself
7856,200070,02 IZMIR,Work With Me
7857,360020,20 TRABZON,Do It Myself


In [111]:
#custdf.info()
custdf.sum(numeric_only=True)

Undercarriage Opportunity                               7438790
Undercarriage Sales                                     3325066
Engine Opportunity                                     15345174
Engine Sales                                           13076342
GET  Opportunity                                        5933717
GET Sales                                               3360649
Drive Train Opportunity                                 9514938
Drive Train Sales                                       7862864
Hydraulics  Opportunity                                14105623
Hydraulics Sales                                        4376568
Filters & Fluids Opportunity                           11567907
Filters & Fluids Sales                                  4739587
Maintenance Parts and Supplies Opportunity              6148534
Maintenance Parts and Supplies Sales                    2058180
Structural, Appearance, and Other Parts Opportunity     2698961
Structural, Appearance, and Other Parts 

In [112]:
# Grouping the measure values by Customer ID to avoid double counting in next steps
# Some Sales are in the negative, this could be taking care of credits ???? 

cust_grp_df = custdf.drop(columns=['Segmentation Rank']).groupby('Customer ID', as_index = False).sum(numeric_only=True)
cust_grp_df
#cust_grp_df.describe()
#cust_grp_df[cust_grp_df['Customer ID'] == 'C531906']  # Example Cust ID verification for the aggregated values

,Customer ID,Undercarriage Opportunity,Undercarriage Sales,Engine Opportunity,Engine Sales,GET Opportunity,GET Sales,Drive Train Opportunity,Drive Train Sales,Hydraulics Opportunity,...,Filters & Fluids Opportunity,Filters & Fluids Sales,Maintenance Parts and Supplies Opportunity,Maintenance Parts and Supplies Sales,"Structural, Appearance, and Other Parts Opportunity","Structural, Appearance, and Other Parts Sales",Labor Opportunity,Labor Sales,Parts Sales,Parts Opportunity
0,310480,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,450650,0,0,8052,2423,3750,0,9776,2000,4101,...,4724,4497,3374,1246,1100,694,15101,3344,13954,34877
2,010020,0,0,102,0,30,0,54,0,108,...,132,0,0,0,0,0,222,0,0,426
3,010030,2971,0,3940,0,1883,0,1810,0,4275,...,2958,0,1880,603,1283,0,8728,0,603,21000
4,010050,0,0,345,0,219,0,6,0,179,...,282,0,173,0,87,0,552,0,0,1291
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7854,C531906,0,0,6653,3417,0,0,0,0,0,...,3652,1193,265,0,27,0,3684,3000,4610,10597
7855,C531912,0,0,0,885,0,0,0,759,0,...,0,205,0,2378,0,14,0,257,4306,0
7856,C531931,0,0,0,146,0,51,0,183,0,...,0,296,0,0,0,0,0,0,676,0
7857,C531969,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [113]:
# custdf[custdf['Customer ID'] == 'C531906'] # Example Cust ID verification

# Vehicledf - total vehicles across customers. Some Customers have more than one name associated and hence needs to be cleaned up
vehicledf.sum(numeric_only=True)
#vehicledf.describe(include="all")

Equipment units    22969
dtype: int64

In [114]:
vehicle_grp_df = vehicledf.drop(columns=['Customer Name']).groupby('Customer ID', as_index = False).sum(numeric_only=True)
vehicle_grp_df

,Customer ID,Equipment units
0,010020,1
1,010030,9
2,010050,2
3,010060,15
4,010070,1
...,...,...
7854,E09981,1
7855,PUZ073,2
7856,PUZ540,35
7857,PUZ630,0


In [115]:
# customerdf[~customerdf['Customer ID'].isin(vehicle_grp_df['Customer ID'])]  
# 1517 Customers in customerdf, but not in vehicle_grp_df

vehicle_grp_df[~vehicle_grp_df['Customer ID'].isin(customerdf['Customer ID'])]
# 1517 Customers in vehicle_grp_df, but not in customerdf

,Customer ID,Equipment units
1409,310480,1
3463,450650,6
6344,E00010,0
6345,E00011,10
6346,E00012,1
...,...,...
7854,E09981,1
7855,PUZ073,2
7856,PUZ540,35
7857,PUZ630,0


In [116]:
vehicledf[vehicledf['Customer ID'] == 'PUZ540']

customerdf[customerdf['Customer ID'].str.contains('ZEN', na=False)]

,Customer ID,City,Segmentation


In [117]:
# salesdf.describe(include="all")
# salesdf.groupby('Customer ID').size()  # groupby count
# salesdf.groupby('Customer ID').filter(lambda x: len(x) > 1)
# salesdf[salesdf['Customer ID'] == 'C526607']
salesdf.sum(numeric_only=True)

Work Order sales          37221008
Over the Counter Sales    42043478
dtype: int64

In [118]:
# salesdf - Sales in different Channels. Some Customers have more than one row and hence needs to be aggregated

sales_grp_df = salesdf.drop(columns=['customer name']).groupby('Customer ID', as_index = False).sum(numeric_only=True)
sales_grp_df
# sales_grp_df.sum(numeric_only=True)
# salesdf[salesdf['Customer ID'] == 'No ID']
sales_grp_df.describe(include="all")

,Customer ID,Work Order sales,Over the Counter Sales
count,5242,5.242000e+03,5.242000e+03
unique,5242,NaN,NaN
top,ZENGEL,NaN,NaN
freq,1,NaN,NaN
mean,NaN,7.100536e+03,8.020503e+03
std,NaN,2.577800e+05,2.927484e+05
min,NaN,0.000000e+00,-2.056000e+03
25%,NaN,0.000000e+00,0.000000e+00
50%,NaN,0.000000e+00,8.000000e+00
75%,NaN,1.006750e+03,4.600000e+02


In [119]:
# customerdf[~customerdf['Customer ID'].isin(sales_grp_df['Customer ID'])]  
# 3978 Customers in customerdf, but not in sales_grp_df

# vehicle_grp_df[~vehicle_grp_df['Customer ID'].isin(sales_grp_df['Customer ID'])]
# 3224 Customers in vehicle_grp_df, but not in sales_grp_df

# sales_grp_df[~sales_grp_df['Customer ID'].isin(customerdf['Customer ID'])]
# 1361 Customers in sales_grp_df, but not in customerdf

sales_grp_df[~sales_grp_df['Customer ID'].isin(vehicle_grp_df['Customer ID'])]
# 607 Customers in sales_grp_df, but not in vehicle_grp_df

,Customer ID,Work Order sales,Over the Counter Sales
31,010940,0,0
64,013940,0,0
65,014010,0,0
100,030320,0,0
113,039997,0,0
...,...,...,...
5222,E09278,0,0
5226,E09329,0,0
5230,E09359,0,0
5238,No ID,18610498,21021728


In [120]:
# All the dataframes are now at the grain of Customer. Combine the information in all to do a comprehensive analysis
# these are the cleaned dataframes that need to be combined
# customerdf, cust_grp_df, vehicle_grp_df, sales_grp_df

# Store dataframes in a list
dflist = [customerdf, cust_grp_df, vehicle_grp_df, sales_grp_df]

all_cust_df = reduce(lambda left, right: pd.merge(left, right, on=['Customer ID'], how="outer"), dflist)
# all_cust_df = customerdf.copy()
# all_cust_df.merge(cust_grp_df, on=['Customer ID'], how="outer").merge(vehicle_grp_df, on=['Customer ID'], how="outer")

# Due to outer join, columns that didn't have values in other dataframes have 'NaN' as the default value
# Change the default 'NaN' value to 0 for numeric columns
# Find the columns with numeric data types

numeric_column_names = all_cust_df.select_dtypes(include=[np.number]).columns.tolist()

# Change 'NaN' values to 0
all_cust_df[numeric_column_names] = all_cust_df[numeric_column_names].fillna(0)

# this is the cleaned dataset
all_cust_df


,Customer ID,City,Segmentation,Undercarriage Opportunity,Undercarriage Sales,Engine Opportunity,Engine Sales,GET Opportunity,GET Sales,Drive Train Opportunity,...,Maintenance Parts and Supplies Sales,"Structural, Appearance, and Other Parts Opportunity","Structural, Appearance, and Other Parts Sales",Labor Opportunity,Labor Sales,Parts Sales,Parts Opportunity,Equipment units,Work Order sales,Over the Counter Sales
0,310480,03 ADANA,No Product Segment Assigned,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,450650,02 IZMIR,Do It Myself,0.0,0.0,8052.0,2423.0,3750.0,0.0,9776.0,...,1246.0,1100.0,694.0,15101.0,3344.0,13954.0,34877.0,0.0,0.0,0.0
2,010020,03 ADANA,Do It Myself,0.0,0.0,102.0,0.0,30.0,0.0,54.0,...,0.0,0.0,0.0,222.0,0.0,0.0,426.0,1.0,0.0,0.0
3,010030,01 ANKARA,Do It Myself,2971.0,0.0,3940.0,0.0,1883.0,0.0,1810.0,...,603.0,1283.0,0.0,8728.0,0.0,603.0,21000.0,9.0,0.0,603.0
4,010050,03 ADANA,Do It Myself,0.0,0.0,345.0,0.0,219.0,0.0,6.0,...,0.0,87.0,0.0,552.0,0.0,0.0,1291.0,2.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9978,PUZ240,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9979,PUZ540,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.0,0.0,0.0
9980,PUZ630,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718.0,632.0
9981,RENTAL,00 ISTANBUL,Do It Myself,3671.0,0.0,11101.0,30350.0,25028.0,0.0,7959.0,...,312.0,2913.0,694.0,30031.0,0.0,78154.0,110390.0,0.0,0.0,0.0


In [121]:
all_cust_df.describe(include = "all")

,Customer ID,City,Segmentation,Undercarriage Opportunity,Undercarriage Sales,Engine Opportunity,Engine Sales,GET Opportunity,GET Sales,Drive Train Opportunity,...,Maintenance Parts and Supplies Sales,"Structural, Appearance, and Other Parts Opportunity","Structural, Appearance, and Other Parts Sales",Labor Opportunity,Labor Sales,Parts Sales,Parts Opportunity,Equipment units,Work Order sales,Over the Counter Sales
count,9983,7859,7859,9983.000000,9983.000000,9983.000000,9983.000000,9983.000000,9983.000000,9983.000000,...,9983.000000,9983.000000,9983.000000,9.983000e+03,9983.000000,9.983000e+03,9.983000e+03,9983.000000,9.983000e+03,9.983000e+03
unique,9983,9,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,ZENGEL,03 ADANA,Do It Myself,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,1836,6016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,745.145748,333.072824,1537.130522,1309.860964,594.382150,336.637183,953.114094,...,206.168486,270.355705,411.803266,3.053285e+03,1125.675949,4.298336e+03,7.287754e+03,2.300811,3.728439e+03,4.211507e+03
std,NaN,NaN,NaN,9232.830276,8699.875526,13414.397738,17114.322540,5259.405004,7444.120373,8506.389219,...,2746.664294,2548.332236,6305.898453,2.246007e+04,11419.751461,5.925264e+04,6.182094e+04,9.499284,1.868211e+05,2.121633e+05
min,NaN,NaN,NaN,0.000000,0.000000,0.000000,-908.000000,0.000000,-2042.000000,0.000000,...,-1382.000000,0.000000,-602.000000,0.000000e+00,0.000000,-2.652000e+03,0.000000e+00,0.000000,0.000000e+00,-2.056000e+03
25%,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00
50%,NaN,NaN,NaN,0.000000,0.000000,175.000000,0.000000,7.000000,0.000000,2.000000,...,0.000000,12.000000,0.000000,4.100000e+02,0.000000,0.000000e+00,8.960000e+02,1.000000,0.000000e+00,0.000000e+00
75%,NaN,NaN,NaN,0.000000,0.000000,880.000000,0.000000,324.000000,0.000000,426.500000,...,0.000000,117.000000,0.000000,2.050000e+03,0.000000,1.860000e+02,4.272500e+03,2.000000,0.000000e+00,2.400000e+01
